In [0]:
%run ./00_config

In [0]:
# ============================================================================
# US-1.09: Parameterized Reference Data Ingestion & Audit Lifecycle (Fixed)
# ============================================================================
import uuid
from pyspark.sql.functions import current_timestamp

# Generate a single unique Batch UUID for this entire execution run
current_batch_id = str(uuid.uuid4())

# ============================================================================
# 2. Universal Parameterized Ingestion Routine with Inline Metadata Injection
# ============================================================================
def ingest_bronze_source(target_table, source_folder_path, file_pattern="*.csv"):
    """
    Ingests raw CSV data using COPY INTO with explicit inline metadata expression mapping
    to guarantee non-null values for _source_file, _ingested_at, and _batch_id.
    """
    print(f"\n🚀 Initiating ingestion for target table: {target_table}")
    print(f"📁 Scanning source directory : {source_folder_path}")
    
    try:
        # We wrap the source inside a SELECT statement that explicitly appends 
        # file lineage and run metadata directly to every incoming row
        copy_query = f"""
        COPY INTO {target_table}
        FROM (
          SELECT 
            *,
            _metadata.file_path AS _source_file,
            current_timestamp() AS _ingested_at,
            '{current_batch_id}' AS _batch_id
          FROM '{source_folder_path}'
        )
        FILEFORMAT = CSV
        PATTERN = '{file_pattern}'
        FORMAT_OPTIONS (
          'header' = 'true',
          'inferSchema' = 'false'
        )
        COPY_OPTIONS (
          'force' = 'false',
          'mergeSchema' = 'true'
        );
        """
        
        # Execute the query and capture execution output metrics dataframe
        copy_result_df = spark.sql(copy_query)
        
        # Extract performance metrics
        metrics = copy_result_df.select("num_inserted_rows", "num_affected_rows").collect()[0]
        rows_loaded  = int(metrics["num_inserted_rows"])
        files_loaded = 1 if rows_loaded > 0 else 0
        
        print(f"✅ Ingestion Complete. Files loaded: {files_loaded} | Rows committed: {rows_loaded}")
        print(f"Pillared auditing metadata (_source_file, _ingested_at, _batch_id) injected inline.")
        
        # Log a structured summary row into your ops.ingestion_audit ledger
        audit_row = [(
            current_batch_id,
            target_table,
            source_folder_path,
            files_loaded,
            rows_loaded
        )]
        
        audit_df = spark.createDataFrame(
            audit_row, 
            schema="batch_id STRING, target_table STRING, source_path STRING, files_loaded LONG, rows_loaded LONG"
        ).withColumn("load_ts", current_timestamp())
        
        audit_df.write.format("delta").mode("append").saveAsTable(table_audit_reconcile)
        print(f"⚙️ Operational audit log successfully recorded in database.")
        
    except Exception as execution_error:
        print(f"❌ FATAL ERROR DURING INGESTION: {str(execution_error)}")
        raise execution_error

# ============================================================================
# 3. Call Routine with Exact Column Counts for Each File Type
# ============================================================================
print(f"🆔 Running Batch ID Workspace Context: {current_batch_id}")

ingest_bronze_source(table_bronze_customers, path_customers)
ingest_bronze_source(table_bronze_accounts, path_accounts)
ingest_bronze_source(table_bronze_branches, path_branches)

print("\n🎉 Reference Data Ingestion with Verified Metadata Pillars Completed Safely.")
